## *A neural network potential for water clusters*


**In a few words -** In this tutorial, we will train and use an interatomic potential based on a neural network, using a DFTB molecular dynamics trajectory of a protonated water cluster (6H<sub>2</sub>O+H<sup>+</sup>) in vacuum as a dataset.  We will then test its transferability using a larger cluster, and bulk water. 

**Tools used -** [aml](https://github.com/MarsalekGroup/aml): active learning of machine-learned potentials, based on [n2p2](https://compphysvienna.github.io/n2p2/), a package for neural network potentials. [cp2k](https://www.cp2k.org/), a general molecular dynamics package. 

**How to use this?** This tutorial uses Python, some familiarity with programming would be beneficial but is not required. Run each code cell in order by clicking "Shift + Enter". Feel free to modify the code, or change queries to familiarize yourself with the workings on the code.

Suggested modifications and exercises are included in <font color=blue> blue</font>.

**Outline:**

1. Loading and visualizing the dataset
2. Selecting an appropriate training set using active learning
3. Training the final model
4. Running molecular dynamics
5. Assessing the quality of the model
6. Transferability to a 16H<sub>2</sub>O+H<sup>+</sup> cluster
7. Transferability to bulk water

**Get started:** Click "Shift-Enter" on the code cells to run! 

**Setup (Google Colab).** Run the next cell first, and again whenever the Colab runtime restarts. It downloads the course files (or uses `TP-MLIP.zip` if you uploaded it in the Files panel on the left), installs n2p2, CP2K and the Python packages we need (1 to 2 minutes the first time), and moves into this tutorial's folder.

In [ ]:
ZIP_URL = 'https://github.com/zacway/TP-MLIP/releases/latest/download/TP-MLIP.zip'

import os
if not os.path.exists('/content/TP-MLIP'):
    # a TP-MLIP.zip uploaded by hand (Files panel on the left) is used instead of downloading it
    if not os.path.exists('/content/TP-MLIP.zip'):
        !curl -Lsf -o /content/TP-MLIP.zip.part "$ZIP_URL" && mv /content/TP-MLIP.zip.part /content/TP-MLIP.zip
    assert os.path.exists('/content/TP-MLIP.zip'), 'Could not download TP-MLIP.zip: upload it with the Files panel on the left, then run this cell again.'
    !unzip -q /content/TP-MLIP.zip -d /content
%run /content/TP-MLIP/colab_setup.py
%cd /content/TP-MLIP/tutorial_02_wclusters

In [ ]:
# We'll start by loading and visualizing the reference DFTB trajectory

%matplotlib inline

import os

base_dir = os.getcwd()

# Some paths and files for the reference DFTB data

ref_dir = base_dir+'/data/cluster/'

ref_trj = ref_dir+'cluster-pos-1.xyz'
ref_frc = ref_dir+'cluster-frc-1.xyz'
ref_vel = ref_dir+'cluster-vel-1.xyz'
ref_ene = ref_dir+'cluster-1.ener'
ref_top = ref_dir+'cluster.pdb'

# Some paths we'll use later on

qbc_dir = base_dir+'/qbc/'
mod_dir = base_dir+'/train/'

# Some conversion constants

au_to_evang = 51.42208619083232
au_to_ev = 27.21138386

# Let's visualize the trajectory. 
# Both linear and angular momenta of the system are removed. 

import mdtraj as mdt
import nglview as nv

nv.show_mdtraj(mdt.load_xyz(ref_trj, top=ref_top, stride=10))

In [ ]:
# Now let's look at some thermodynamical quantities over the course of the trajectory

os.chdir(base_dir)

import numpy as np

def compute_time(thermo, n_cpus):
    t_min = np.sum(thermo[6,:])/60
    n_steps = int(thermo[-1,0])
    rate = int(n_steps/t_min/n_cpus)
    print('The full simulation took {:02.2f} minutes using {} CPU, for a total of {} MD steps'\
          ', corresponding to an average compute rate of {} MD steps per minute per CPU. '\
          .format(t_min, n_cpus, n_steps, rate))

import matplotlib.pyplot as plt
    
def plot_thermo(thermo):
    fig, axs = plt.subplots(2, 1)
    
    for ax in axs.flat:
        ax.set_xlim([0.0, 1e-3*thermo[-1,1]])
        
    axs[0].set_ylabel(r'$E_{p}$ (eV)')
    axs[0].plot(1e-3*thermo[::100,1], au_to_ev*thermo[::100,4], color='firebrick')

    axs[1].set_xlabel(r't (ps)')
    axs[1].set_ylabel(r'T (K)')
    axs[1].plot(1e-3*thermo[::100,1], thermo[::100,3], color='dodgerblue')
    axs[1].hlines(np.average(thermo[:,3]), 1e-3*thermo[0,1], 1e-3*thermo[-1,1], alpha=0.5, color='gray')
    
ref_thermo = np.loadtxt(ref_ene)

compute_time(ref_thermo, 24)

plot_thermo(ref_thermo)

In [ ]:
# We'll start performing active learning: an optimal training set
# will be constructed using the "Query by Committee" algorithm. 

os.chdir(base_dir)

import aml

def read_structures(trj_file, frc_file, stride):
    # stride through trajectory

    print('Reading structures')
    print('------------------')
    print()
    print(f'Stride: {stride}')
    print()

    frames = aml.read_frames_cp2k(fn_positions=trj_file, fn_forces=frc_file)
    structures = aml.Structures.from_frames(frames, stride=stride, probability=1.0)

    print(f'{len(structures):} structures kept')
    print()

    return structures

import sys
import fileinput

def set_nn(n_hidden, n_neurons, activ):
    for line in fileinput.input(qbc_dir+'input.nn', inplace=True):
        if line.strip().startswith('global_hidden_layers_short'):
            line = 'global_hidden_layers_short      '+str(n_hidden)+' \n'
        if line.strip().startswith('global_nodes_short'):
            line = 'global_nodes_short              '+str(n_neurons)+' \n'
        if line.strip().startswith('global_activation_short'):
            line = 'global_activation_short         '+str(activ)+' \n'
        sys.stdout.write(line)

# Prepare the neural network architecture
# Edit the three following lines if you want to experiment with different layouts

n_hidden = 2
n_neurons = '5 5 '
activ = 't t l'

# If you've already run a QbC training once and you want to train a new model, 
# remove the "iteration-x" folders and the results.shelf.* files first, to avoid any issues: 
# !rm -rf qbc/iteration-* qbc/results.shelf*

n_iterations = 15
n_members = 8

set_nn(n_hidden, n_neurons, activ)

# Settings - n2p2 constructor
kwargs_model = dict(
    elements = ['O', 'H'],
    n = n_members,
    fn_template = qbc_dir+'input.nn',
    #exclude_triples = [['O', 'O', 'O']],
    n_tasks = 8,
    n_core_task = 1,
    remove_output = True
)

# Structures to select from
structures = read_structures(ref_trj, ref_frc, 1)

# Parameters of the query by committee algorithm
qbc = aml.QbC(
    structures = structures,
    cls_model = aml.N2P2,
    kwargs_model = kwargs_model,
    n_train_initial = 20,
    n_add = 20,
    n_epoch = 15,
    n_iterations = n_iterations,
    n_candidate = 500,
    fn_results = 'results.shelf',
    fn_restart = None
)

os.chdir(qbc_dir)
qbc.run()
os.chdir(base_dir)

In [ ]:
# Plot the initial and final training points, 
# along the potential energy trajectory. 

import shelve

def plot_training(thermo, n_iterations):
    fig, ax = plt.subplots(1, 1, figsize=(14,4))
    
    ax.set_xlim([0.0, 1e-3*thermo[-1,1]])
    
    ax.set_xlabel(r't (ps)')    
    ax.set_ylabel(r'$E_{p}$ (eV)')
    ax.plot(1e-3*thermo[:,1], au_to_ev*thermo[:,4], alpha=0.5, color='gray')
    
    with shelve.open('qbc/results.shelf') as res:
        ts_ini = res['iteration-001']
        ts_fin = res['iteration-'+f'{n_iterations:03d}']

        for ind in ts_fin['idx_train']:
            ax.scatter(1e-3*thermo[5*ind,1], au_to_ev*thermo[5*ind,4], marker='o', s=6.0, alpha=0.8, color='dodgerblue')
            
        for ind in ts_ini['idx_train']:
            ax.scatter(1e-3*thermo[5*ind,1], au_to_ev*thermo[5*ind,4], marker='o', s=6.0, alpha=0.8, color='firebrick')

plot_training(ref_thermo, n_iterations)

In [ ]:
# Using the last training set, let's now optimize our neural network model
# with a greater number of epochs. This will constitute our final model. 

# If you've already trained a model once and you want to train a new model, 
# remove the "final_training" folder first, to avoid any issues: 
# !rm -rf train/final_training

os.chdir(base_dir)

# Load our final training set
print('Loading training set... ', end='', flush=True)
structures = aml.Structures.from_file(qbc_dir+f'iteration-{n_iterations:03d}/train-000/input.data')
print('done.')
print()

# Initialize the model, perform training, save the final model. 
os.chdir(mod_dir)
print('Creating blank NNP model... ', end='', flush=True)
# Settings - n2p2 constructor
kwargs_model = dict(
    elements = ['O', 'H'],
    n = n_members,
    fn_template = qbc_dir+'input.nn',
    # exclude_triples = [['O', 'O', 'O']],
    n_tasks = 4,
    n_core_task = 1
)
n2p2 = aml.N2P2(dir_run=mod_dir+'final_training', **kwargs_model)
print('done.')
print()

print('Training all member models... ', end='', flush=True)
n2p2.train(structures, n_epoch=50)
print('done.')
print()

print('Saving all member models... ', end='', flush=True)
n2p2.save_model()
print('done.')
print()
os.chdir(base_dir)

In [ ]:
# Let's plot learning curves for each member of the final model. 

def plot_optimization(opt_dir, n_members):
    fig, axs = plt.subplots(2, 1, figsize=(8,8))
    
    c_list = iter(plt.cm.rainbow(np.linspace(0,1,n_members)))

    for co in range(n_members):
        lea_data = np.loadtxt(opt_dir+'/train-'+f'{co:03d}'+'/learning-curve.out', usecols=(0,11,12))
        col = next(c_list)
        axs[0].plot(lea_data[:,0], au_to_evang*lea_data[:,1], alpha=0.8, color=col, label='Member '+str(co))
        axs[1].plot(lea_data[:,0], au_to_evang*lea_data[:,2], alpha=0.8, color=col)
    
    for ax in axs.flat:
        ax.set_xlim([lea_data[0,0], lea_data[-1,0]])
        ax.set_yscale('log')
    
    axs[1].set_xlabel(r'Epoch')    
    axs[0].set_ylabel(r'MAE train forces (eV/$\AA$)')
    axs[1].set_ylabel(r'MAE test forces (eV/$\AA$)')
    axs[0].legend()
    
plot_optimization(mod_dir+'final_training', n_members)

**Now that we have a model, let's run some MD!**

* <font color=blue>If you've modified the number of committee members, adjust the cp2k input script ("nnp_md/cluster.inp", open it from the Files panel on the left)</font>
* Run the next cell to start the CP2K MD simulation in the "nnp_md" subfolder. It prints the latest line of the energy file every 15 s. Once it reports "Done", you can proceed.

In [ ]:
run_cp2k('cluster.inp', base_dir+'/nnp_md')

In [ ]:
# Time to visualize our newly-generated trajectory. 

os.chdir(base_dir)

nnp_dir = base_dir+'/nnp_md/'

nnp_trj = nnp_dir+'cluster-pos-1.xyz'
nnp_frc = nnp_dir+'cluster-frc-1.xyz'
nnp_vel = nnp_dir+'cluster-vel-1.xyz'
nnp_ene = nnp_dir+'cluster-1.ener'
nnp_top = nnp_dir+'cluster.pdb'

nnp_thermo = np.loadtxt(nnp_ene)

compute_time(nnp_thermo, 1)

plot_thermo(nnp_thermo)

nv.show_mdtraj(mdt.load_xyz(nnp_trj, top=nnp_top, stride=10))

If you see a water molecule flying away from the cluster, this can be due to the DFTB model, which _significantly_ underestimates water molecule interactions. 

In [ ]:
# We'll now compute a few properties, to quantitatively assess
# the quality of our model. 

import aml.score as mlps

dt_trj = 2.5 # Time between MD frames, in fs

# Loading position data
ref_pos = mlps.load_with_cell(ref_trj, top=ref_top)
nnp_pos = mlps.load_with_cell(nnp_trj, top=nnp_top)

# Perform RDF scoring
mlps.run_rdf_test(ref_pos, nnp_pos)

# Q's: why is the O-O error larger? Is it surprising to get such a good agreement for a
# single molecule? 

In [ ]:
# Loading velocity data
ref_atv = mlps.load_with_cell(ref_vel, top=ref_top)
nnp_atv = mlps.load_with_cell(nnp_vel, top=nnp_top)

# Perform VDOS scoring
mlps.run_vdos_test(ref_atv, dt_trj, nnp_atv, dt_trj)

The model seems to be fairly satisfying. Can we use it on a larger cluster, say for instance 16H<sub>2</sub>O+H<sup>+</sup>? Run the next cell to start an MD simulation in the "big_cluster" subfolder of "tutorial_02_wclusters".

We have DFTB reference data for this larger cluster. We will therefore be able to assess the accuracy based on the same criteria as before (radial distribution functions and vibrational density of states): 

In [ ]:
run_cp2k('big_cluster.inp', base_dir+'/big_cluster')

In [ ]:
# First visualize the reference trajectory

ref_big_dir = base_dir+'/data/big_cluster/'

ref_big_trj = ref_big_dir+'big_cluster-pos-1.xyz'
ref_big_frc = ref_big_dir+'big_cluster-frc-1.xyz'
ref_big_vel = ref_big_dir+'big_cluster-vel-1.xyz'
ref_big_ene = ref_big_dir+'big_cluster-1.ener'
ref_big_top = ref_big_dir+'big_cluster.pdb'

# Visualize the reference trajectory
ref_big_view = mdt.load_xyz(ref_big_trj, top=ref_big_top)
nv.show_mdtraj(ref_big_view)

In [ ]:
# Visualize the NNP trajectory

nnp_big_dir = base_dir+'/big_cluster/'

nnp_big_trj = nnp_big_dir+'big_cluster-pos-1.xyz'
nnp_big_frc = nnp_big_dir+'big_cluster-frc-1.xyz'
nnp_big_vel = nnp_big_dir+'big_cluster-vel-1.xyz'
nnp_big_ene = nnp_big_dir+'big_cluster-1.ener'
nnp_big_top = nnp_big_dir+'big_cluster.pdb'

# Visualize the NNP trajectory
nnp_big_view = mdt.load_xyz(nnp_big_trj, top=nnp_big_top)
nv.show_mdtraj(nnp_big_view)


In [ ]:
# Then assess the accuracy of the model 

# Loading position data
ref_big_pos = mlps.load_with_cell(ref_big_trj, top=ref_big_top)
nnp_big_pos = mlps.load_with_cell(nnp_big_trj, top=nnp_big_top)

# Perform RDF scoring
mlps.run_rdf_test(ref_big_pos, nnp_big_pos)

# Loading velocity data
ref_big_atv = mlps.load_with_cell(ref_big_vel, top=ref_big_top)
nnp_big_atv = mlps.load_with_cell(nnp_big_vel, top=nnp_big_top)

# Perform VDOS scoring
mlps.run_vdos_test(ref_big_atv, dt_trj, nnp_big_atv, dt_trj)

<font color=blue>The scores seem almost reasonnable, but from a visual inspection of the trajectories, do you think that the dynamics are well reproduced? Draw conclusions both for accuracy criteria and for transferability of such models. </font>

**Is our model transferable enough for bulk water?**

We can try running bulk water dynamics in the _NpT_ ensemble using our model. Run the next cell to start the simulation in the "bulk" subfolder of this tutorial's main directory.

NB: this is neutral, unprotonated water. 

In [ ]:
run_cp2k('bulk.inp', base_dir+'/bulk')

In [ ]:
# Post-processing the bulk water MD data: we'll only look at the density time-evolultion here. 

def plot_density(cell_data, n_w_mol):
    n_avo = 6.022140857e23
    mw_w = 18.01528 # water molar mass
    ang_to_cm = 1.0e-8
    
    fig, ax = plt.subplots(1, 1)
    
    ax.set_xlim([0.0, 1e-3*cell_data[-1,1]])
    
    rho_data = n_w_mol*mw_w/n_avo/(cell_data[:,2]*ang_to_cm)**3
    
    ax.set_xlabel(r't (ps)')
    ax.set_ylabel(r'$\rho$ (g/cm$^3$)')
    ax.plot(1e-3*cell_data[::100,1], rho_data[::100], color='firebrick')
    ax.hlines(np.average(rho_data[::100]), 1e-3*cell_data[0,1], 1e-3*cell_data[-1,1], alpha=0.5, color='gray')

nnp_cell = np.loadtxt(base_dir+'/bulk/bulk-1.cell')

plot_density(nnp_cell, 64)

bulk_trj = mdt.load_dcd(base_dir+'/bulk/bulk-pos-1.dcd', top=base_dir+'/bulk/bulk.pdb')
nv.show_mdtraj(bulk_trj)

<font color=blue>What is happening? Keep in mind that CP2K outputs unwrapped xyz coordinates, and that nglviewer doesn't handle periodic boundary conditions. </font>

It seems like direct information from the bulk is needed. There is a reference DFTB trajectory for a system of 64 water molecules stored in the "data/bulk" subdirectory of this tutorial's main folder. Below, we'll monitor convergence to the equilibrium P=1atm, T=300K density:

In [ ]:
dftb_cell = np.loadtxt(base_dir+'/data/bulk/bulk-1.cell')

plot_density(dftb_cell, 64)

As you can see, this DFTB parameterization results in a _significantly_ overstructured water model. It's always important to check in the first place if our reference data is accurate compared to experimental evidence: our machine learning model can only try to mimic the reference. In the next tutorial, we will extract published neural network potentials for complex aqueous systems. 